# 02 — Train YOLOv11

Trains YOLOv11-Medium on the BDD100K clear-weather training split.
Checkpoints are saved every 10 epochs to `checkpoints/yolov11/`.

**Prerequisites:** run `01_setup_and_data.ipynb` first.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv('../.env')

import torch
from ultralytics import YOLO
from src.train_utils import setup_logging

logger = setup_logging()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

In [ ]:
# Load model
model = YOLO('yolo11m.pt')  # downloads COCO pretrained weights on first run

In [ ]:
# TODO: Adjust hyperparameters as needed before launching training.
results = model.train(
    data='../configs/dataset.yaml',
    epochs=100,
    batch=16,
    imgsz=640,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,
    cos_lr=True,
    device=DEVICE,
    project='../checkpoints/yolov11',
    name='run',
    save_period=10,
    exist_ok=True,
    plots=True,
)
print('Training complete.')
print('Best mAP@50:', results.results_dict.get('metrics/mAP50(B)'))

In [ ]:
# Save final best weights summary
import json
from pathlib import Path

summary = {
    'model': 'yolov11',
    'best_map50': results.results_dict.get('metrics/mAP50(B)'),
    'best_map50_95': results.results_dict.get('metrics/mAP50-95(B)'),
    'epochs': 100,
}
out = Path('../checkpoints/yolov11/training_summary.json')
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(summary, indent=2))
print('Summary saved to', out)